# Qwen Baseline Parity Launcher

Thin Kaggle notebook for parity runs of the extracted NVARC/Qwen baseline.

Attach three inputs before running:
- ARC competition input
- model input
- a code dataset containing `ARC-AGI1/qwen_baseline/`


In [ ]:
PARITY_KEYS = [
    "00576224",
    "009d5c81",
]
END_TIME_HOURS = 6
NPROCS = 4

COMP_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
# Set this to the root of the attached code dataset that contains ARC-AGI1/qwen_baseline.
CODE_DATASET_ROOT = "/kaggle/input/nvarc-code"
WORK_ROOT = "/kaggle/working/nvarc_code"
OUTPUT_DIR = "/kaggle/working/inference_outputs"


In [ ]:
!pip install unsloth==2025.9.7 unsloth_zoo==2025.9.9 numpy==2.2.6 matplotlib==3.10.6 scikit-learn==1.7.2
!pip install --upgrade --no-deps --no-build-isolation flash-attn


In [ ]:
patch = b'@@ -272,7 +277,7 \\r\\n \\r\\n         # Mistral Nemo 12b has weird dimensions\\r\\n         if attention_size != hidden_size:\\r\\n-            self.temp_O = torch.empty((1, bsz, hidden_size), dtype = dtype, device = device)\\r\\n+            self.temp_O = torch.empty((bsz, 1, hidden_size), dtype = dtype, device = device)\\r\\n         else:\\r\\n             self.temp_O = self.temp_QA[1][:,:,:hidden_size]\\r\\n         pass\\r\\n@@ -333,52 +338,12 \\r\\n     Kn = self.paged_attention_K[:kv_seq_len].permute(1, 2, 0, 3)\\r\\n     Vn = self.paged_attention_V[:kv_seq_len].permute(1, 2, 0, 3)\\r\\n \\r\\n-    # Handle sliding windows\\r\\n-    sliding_window = getattr(self.config, \"sliding_window\", None)\\r\\n-    if sliding_window is not None and kv_seq_len > sliding_window:\\r\\n-        # From https://github.com/huggingface/transformers/blob/main/src/transformers/models/mistral/modeling_mistral.py#L193\\r\\n-        slicing_tokens = 1 - sliding_window\\r\\n-        Knn = Kn[:, :, slicing_tokens:, :]#.contiguous()\\r\\n-        Vnn = Vn[:, :, slicing_tokens:, :]#.contiguous()\\r\\n-    else:\\r\\n-        Knn, Vnn = Kn, Vn\\r\\n-    pass\\r\\n+    Qnn = Qn.transpose(1, 2)\\r\\n+    Knn = Kn.transpose(1, 2)\\r\\n+    Vnn = Vn.transpose(1, 2)\\r\\n \\r\\n-    # when qlen==vlen and attn_mask is None, we should use causal attention\\r\\n-    Q_len = Qn.shape[-2]\\r\\n-    K_len = Knn.shape[-2]\\r\\n-    if attention_mask is None and Q_len == K_len:\\r\\n-        is_causal = True\\r\\n-    else:\\r\\n-        is_causal = False\\r\\n+    A = flash_attn_func(Qnn, Knn, Vnn)\\r\\n \\r\\n-    # Grouped query attention\\r\\n-    _, _, cached_len, _ = Knn.shape\\r\\n-    if bsz == 1 or not SDPA_HAS_GQA and n_groups != 1:\\r\\n-        Knn = Knn[:, :, None, :, :].expand(bsz, n_kv_heads, n_groups, cached_len, head_dim)\\r\\n-        Vnn = Vnn[:, :, None, :, :].expand(bsz, n_kv_heads, n_groups, cached_len, head_dim)\\r\\n-        Knn = Knn.reshape(bsz, n_heads, cached_len, head_dim)\\r\\n-        Vnn = Vnn.reshape(bsz, n_heads, cached_len, head_dim)\\r\\n-    pass\\r\\n-    # else:\\r\\n-    #     Knn, Vnn = Knn, Vnn\\r\\n-    # pass\\r\\n-\\r\\n-    # Attention\\r\\n-    if bsz == 1:\\r\\n-        Qn *= self.scalar # See https://github.com/ggerganov/llama.cpp/issues/7805#issuecomment-2153349963\\r\\n-        # It seems like doing (Q * scalar) @ K is better than (Q @ K) * scalar to stop overflows\\r\\n-        A = torch_matmul(Qn, Knn.transpose(2, 3), out = self.attention[:,:,:,:cached_len])\\r\\n-        # if attention_mask is not None: A += attention_mask # Must add attention_mask for batched\\r\\n-        A[:] = torch_nn_functional_softmax(A, dim = -1, dtype = torch.float32)#.to(A.dtype)\\r\\n-        A = torch_matmul(A, Vnn, out = Qn)\\r\\n-    else:\\r\\n-        if SDPA_HAS_GQA:\\r\\n-            A = scaled_dot_product_attention(Qn, Knn, Vnn, attn_mask = attention_mask, is_causal = is_causal, enable_gqa = True)\\r\\n-        else:\\r\\n-            A = scaled_dot_product_attention(Qn, Knn, Vnn, attn_mask = attention_mask, is_causal = is_causal)\\r\\n-    pass\\r\\n-    A = A.transpose(1, 2)\\r\\n     A = A.reshape(bsz, 1, attention_size)\\r\\n     A = fast_linear_forward(self.o_proj, A, out = self.temp_O)\\r\\n     return A, (Kn, Vn)\\r\\n'
with open('/kaggle/working/qwen3.patch', 'wb') as f:
    f.write(patch)
!patch --binary /usr/local/lib/python3.11/dist-packages/unsloth/models/qwen3.py /kaggle/working/qwen3.patch


In [ ]:
import json
import os
import pathlib
import shutil

code_root = pathlib.Path(CODE_DATASET_ROOT)
if not code_root.exists():
    raise FileNotFoundError(f'Code dataset root not found: {code_root}')

candidate_paths = [
    code_root / 'ARC-AGI1' / 'qwen_baseline',
    code_root / 'qwen_baseline',
]
src = None
for p in candidate_paths:
    if p.exists():
        src = p
        break
if src is None:
    raise FileNotFoundError(f'Could not find qwen_baseline under {code_root}')

dst = pathlib.Path(WORK_ROOT)
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst)
print('cwd =', os.getcwd())
print('files =', sorted(x.name for x in dst.iterdir()))

assert pathlib.Path(COMP_PATH).exists(), f'Missing competition input: {COMP_PATH}'
assert pathlib.Path(MODEL_PATH).exists(), f'Missing model input: {MODEL_PATH}'
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

with open('/kaggle/working/parity_keys.txt', 'w') as f:
    for key in PARITY_KEYS:
        f.write(f'{key}\n')


In [ ]:
import subprocess
import time

cmd = [
    'python', 'starter.py',
    '--test-path', COMP_PATH,
    '--model-path', MODEL_PATH,
    '--output-dir', OUTPUT_DIR,
    '--keys-file', '/kaggle/working/parity_keys.txt',
    '--nprocs', str(NPROCS),
    '--end-time', str(time.time() + END_TIME_HOURS * 3600),
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import os
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
print('output files:')
for name in sorted(os.listdir(out_dir)):
    print(name)
